# Module `solver_input.py`

Ce notebook presente le contrat d'entree partage par **tous** les algorithmes de resolution du projet.

Principe : un solveur ne doit jamais lire directement un `GraphInstance` ou un `DynamicGraphSnapshot`. Il consomme uniquement un `SolverInput`. Cette couche d'adaptation, proche d'un objet de transfert de donnees [1], garde les algorithmes independants de la provenance des donnees : statique, dynamique, benchmark, test unitaire ou futur import de fichier.

Le role de `solver_input.py` est donc petit mais central : transformer l'etat du reseau en une forme minimale, stable et optimisable.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_network import DynamicNetworkSimulator
from cesipath.graph_generator import GraphGenerator
from cesipath.models import GraphGenerationConfig
from cesipath.solver_input import SolverInput, build_dynamic_solver_input, build_static_solver_input

## 1. Les champs du contrat

`SolverInput` est une dataclass frozen au sens des dataclasses Python [2]. Ses champs :

| Champ | Type | Role |
|---|---|---|
| `cost_matrix` | `list[list[float]]` | Matrice complete deja metrique |
| `depot` | `int` | Index du sommet depot |
| `demands` | `dict[int, int]` | Demande par client |
| `vehicle_capacity` | `int` | Capacite maximale d'un vehicule |
| `shortest_paths` | `dict[(i,j), path]` | Chemins reels derriere la matrice complete |
| `source` | `str` | `static` ou `dynamic` |
| `dynamic_step` | `int | None` | Numero de tour si dynamique |

La matrice est deja complete : un algorithme peut lire `cost_matrix[i][j]` sans verifier si l'arete existe physiquement. Cette hypothese est ce qui simplifie tous les operateurs de voisinage, les heuristiques et les benchmarks.

`shortest_paths` garde le lien avec le monde reel. Une solution peut etre optimisee sur la matrice complete, puis deployee en redescendant chaque saut abstrait vers sa vraie sequence de sommets.


## 2. Construction statique

`build_static_solver_input(instance)` extrait les donnees de l'instance statique :

- la matrice `completed_costs`, obtenue par Dijkstra sur les aretes residuelles non `FORBIDDEN` ;
- les `completed_paths` associes ;
- le depot, les demandes et la capacite ;
- `source="static"`, `dynamic_step=None`.

Le solveur ne voit ni `base_edges`, ni `residual_edges`, ni les statuts d'aretes. C'est volontaire : au moment de resoudre un VRP statique, la seule question utile est "combien coute le passage de `i` a `j` dans le reseau disponible ?".

Cette fonction est aussi le point d'entree des notebooks d'algorithmes. Tous partent d'une instance generee, puis construisent ce contrat avant d'appeler GRASP, SA, tabou ou genetique.


In [ ]:
config = GraphGenerationConfig(node_count=8, seed=42)
instance = GraphGenerator(config).generate()

static_input = build_static_solver_input(instance)
print("Type         :", type(static_input).__name__)
print("source       :", static_input.source)
print("depot        :", static_input.depot)
print("capacity     :", static_input.vehicle_capacity)
print("demandes     :", static_input.demands)
print("nb sommets   :", len(static_input.cost_matrix))
print("chemin 0->3  :", static_input.shortest_paths.get((0, 3)))

## 3. Construction dynamique

`build_dynamic_solver_input(instance, snapshot)` produit la meme structure, mais les couts et chemins viennent du **snapshot** courant :

- `cost_matrix = snapshot.completed_costs` ;
- `shortest_paths = snapshot.completed_paths` ;
- `source="dynamic"` ;
- `dynamic_step = snapshot.step`.

Le reste reste identique : depot, demandes et capacite viennent de l'instance. Dans ce projet, les commandes clients ne changent pas pendant la simulation ; seul le reseau evolue.

**Pourquoi garder `instance` en parametre ?** Parce que le snapshot ne duplique pas tout. Il porte l'etat dynamique du reseau, tandis que l'instance porte les donnees stables du probleme. La fabrique assemble les deux pour produire le contrat complet attendu par les solveurs.


In [ ]:
simulator = DynamicNetworkSimulator(instance, seed=42)
snapshot = simulator.advance(simulator.initialize_snapshot())

dynamic_input = build_dynamic_solver_input(instance, snapshot)
print("source       :", dynamic_input.source)
print("dynamic_step :", dynamic_input.dynamic_step)
print("1ere ligne   :", dynamic_input.cost_matrix[0])

## 4. Usage dans les algorithmes

Tous les algorithmes de `cesipath.algorithms` suivent la meme signature :

```python
solution = algo(solver_input, **hyperparams)
```

ou `algo` peut etre `grasp`, `simulated_annealing`, `tabu_search`, `genetic_algorithm`.

Le solveur consulte uniquement :

- `cost_matrix` pour evaluer les deplacements ;
- `demands` et `vehicle_capacity` pour verifier les tournees ;
- `depot` pour fermer les routes.

Il n'a pas besoin de savoir si la matrice vient du graphe initial ou du tour dynamique 12. Cette indifference suit le principe de separation des responsabilites [3] et devient tres utile pour les benchmarks : on peut comparer les memes algorithmes dans plusieurs contextes sans changer leur code.


In [ ]:
from cesipath.algorithms import grasp

solution = grasp(static_input, max_iterations=5, rcl_alpha=0.2, seed=42)
print("Nb tournees :", solution.route_count)
print("Cout total  :", round(solution.total_cost, 2))
print("Tournees    :", solution.routes)

## 5. Pourquoi un type frozen ?

`SolverInput` est immuable. Un solveur ne peut pas remplacer la matrice, modifier les demandes ou changer la capacite en cours d'execution.

Consequences :

- un benchmark peut lancer plusieurs algorithmes sur le meme `SolverInput` sans contamination croisee ;
- les tests sont plus fiables, car l'entree reste identique avant et apres l'appel ;
- la distinction entre donnees d'entree et solution produite reste nette.

L'immuabilite ne rend pas les listes internes profondement immuables, mais elle fixe le contrat au niveau de la dataclass : un algorithme bien ecrit traite `SolverInput` comme une lecture seule.

A retenir : `solver_input.py` est l'adaptateur entre monde reseau et monde optimisation. Plus ce contrat est petit, plus les algorithmes restent reutilisables.

---

## References

[1] **Fowler, M. (2002).** *Patterns of Enterprise Application Architecture.* Addison-Wesley.

[2] **Python Software Foundation.** *dataclasses - Data Classes.* Python documentation. https://docs.python.org/3/library/dataclasses.html

[3] **Parnas, D. L. (1972).** *On the criteria to be used in decomposing systems into modules.* Communications of the ACM, 15(12), 1053-1058. https://doi.org/10.1145/361598.361623
